# 🦷 OralScan AI - Free Cloud Backend (Google Colab)
This notebook runs your entire **FastAPI + Auth + YOLO + DenseNet-BERT AI Engine** on Google's free GPU/RAM and exposes a public HTTPS URL using Cloudflare Tunnel.

### Steps:
1. In Colab menu: **Runtime -> Change runtime type -> T4 GPU** (or CPU).
2. Run all cells in order.

In [ ]:
# 1. Clone repository or pull latest updates
import os
if not os.path.exists('Oral_Lesion_Detection'):
    !git clone https://github.com/marttpq/Oral_Lesion_Detection.git
    %cd Oral_Lesion_Detection
else:
    %cd /content/Oral_Lesion_Detection
    !git pull origin main

In [ ]:
# 2. Install all Python Dependencies
!pip install -q -r backend/requirements.txt -r ai-engine/requirements.txt google-auth aiosqlite

In [ ]:
# 3. Verify Model Weights
from pathlib import Path

yolo_weight = Path('ai-engine/yolo-service/yolov7.pt')
bert_weight = Path('ai-engine/densenet-bert-service/best_text.pth')

if not yolo_weight.exists() or not bert_weight.exists():
    print("⚠️ Model weights not found in Git repository. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    print("Copy your weights from Drive into ai-engine/yolo-service/ and ai-engine/densenet-bert-service/")
else:
    print("✅ All model weights found!")

In [ ]:
# 4. Install Cloudflare Tunnel (100% Free, no account needed)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
# 5. Check routes and start Backend with Cloudflare Tunnel
import subprocess, time, re

# Verify all endpoints are properly loaded
from main import app
print("✅ FastAPI loaded successfully! Registered routes:")
for route in app.routes:
    methods = getattr(route, 'methods', None)
    if methods:
        print(f"   {list(methods)} {route.path}")

# Start FastAPI backend on port 8000
backend_proc = subprocess.Popen(["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"])
time.sleep(3)

# Launch Cloudflare Tunnel and grab the generated public URL
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000"], stderr=subprocess.PIPE, text=True)

print("\n⏳ Connecting Cloudflare Tunnel...")
for line in tunnel_proc.stderr:
    if "trycloudflare.com" in line:
        match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if match:
            public_url = match.group(0)
            print("\n" + "="*60)
            print(f"🎉 PUBLIC BACKEND URL: {public_url}")
            print("="*60)
            print("\n👉 If changed, update Vercel VITE_API_URL and redeploy.")
            break